In [1]:
from pyspark.sql import SparkSession
from delta import *

warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Explorar silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
spark.sql("SHOW DATABASES").show()

+------------------+
|         namespace|
+------------------+
|           default|
|              demo|
|silver_electronics|
+------------------+



In [3]:
spark.sql("SHOW TABLES IN silver_electronics").show()

+------------------+------------------+-----------+
|         namespace|         tableName|isTemporary|
+------------------+------------------+-----------+
|silver_electronics|   amazon_products|      false|
|silver_electronics|    amazon_ratings|      false|
|silver_electronics|            prices|      false|
|silver_electronics|prices_amazon_link|      false|
+------------------+------------------+-----------+



In [4]:
for t in ["prices", "amazon_ratings", "amazon_products", "prices_amazon_link"]:
    info = spark.sql(f"DESCRIBE TABLE EXTENDED silver_electronics.{t}").toPandas()
    linhas = info[info["col_name"].isin(["Type", "Provider", "Location"])]
    print(t)
    print(linhas[["col_name", "data_type"]].to_string(index=False, header=False), "\n")

prices
    Type                                                EXTERNAL
Location hdfs://hdfs-nn:9000/silver/silver_electronics.db/prices
Provider                                                   delta 

amazon_ratings
    Type                                                        EXTERNAL
Location hdfs://hdfs-nn:9000/silver/silver_electronics.db/amazon_ratings
Provider                                                           delta 

amazon_products
    Type                                                         EXTERNAL
Location hdfs://hdfs-nn:9000/silver/silver_electronics.db/amazon_products
Provider                                                            delta 

prices_amazon_link
    Type                                                            EXTERNAL
Location hdfs://hdfs-nn:9000/silver/silver_electronics.db/prices_amazon_link
Provider                                                               delta 



In [5]:
spark.sql("DESCRIBE silver_electronics.prices").toPandas()

,col_name,data_type,comment
0,product_id,string,None
1,name,string,None
2,brand,string,None
3,manufacturer_number,string,None
4,categories,string,None
5,asins,string,None
6,merchant,string,None
7,amount_min,double,None
8,amount_max,double,None
9,is_sale,boolean,None


In [6]:
spark.sql("DESCRIBE silver_electronics.amazon_products").toPandas()

,col_name,data_type,comment
0,asin,string,None
1,title,string,None
2,brand,string,None
3,main_cat,string,None
4,sub_category,string,None
5,category_path,string,None
6,price,double,None
7,release_date,date,None
8,sales_rank,int,None


In [7]:
spark.sql("DESCRIBE silver_electronics.amazon_ratings").toPandas()

,col_name,data_type,comment
0,asin,string,None
1,reviewer_id,string,None
2,rating,int,None
3,review_date,date,None


In [8]:
spark.sql("DESCRIBE silver_electronics.prices_amazon_link").toPandas()

,col_name,data_type,comment
0,product_id,string,None
1,asin,string,None
2,match_type,string,None


In [9]:
spark.sql("""
    SELECT 'prices' AS tabela, COUNT(*) AS linhas FROM silver_electronics.prices
    UNION ALL SELECT 'amazon_ratings', COUNT(*) FROM silver_electronics.amazon_ratings
    UNION ALL SELECT 'amazon_products', COUNT(*) FROM silver_electronics.amazon_products
    UNION ALL SELECT 'prices_amazon_link', COUNT(*) FROM silver_electronics.prices_amazon_link
""").toPandas()

,tabela,linhas
0,prices,33695
1,amazon_ratings,20568091
2,amazon_products,756061
3,prices_amazon_link,575


In [10]:
spark.sql("SELECT * FROM silver_electronics.prices LIMIT 10").toPandas()

,product_id,name,brand,manufacturer_number,categories,asins,merchant,amount_min,amount_max,is_sale,is_discount,discount_pct,date_seen
0,AVpiSS1A1cnluZ0-LyAY,Transcend - JetDrive 420 480GB Internal Serial...,Transcend,TS480GJDM420,"Computers,Hard Drives,Internal Drives,Solid St...","B00JQXT82I,B00QXMI1HI",Best Buy,270.99,270.99,True,False,0.00,2014-09-28
1,AV13iAUYGV-KLJ3aka9M,"Samsung - 50 Class (49.5"" Diag.) - LED - 1080p...",Samsung,UN50M5300FXZA,"LCD TVs,Samsung TVs,Electronics,Shop TVs by Ty...",B0725FX4RF,Best Buy,431.99,431.99,True,False,0.00,2017-08-30
2,AVpgm2X2ilAPnD_xu0wB,Virtually Invisible 891 In-Wall Speakers (Pair),Bose,742896-0200,"In-Wall & In-Ceiling Speakers,In-Wall Speakers...",B013WQIDSY,B&H Photo,599.00,599.00,False,False,0.00,2016-05-05
3,AV4GSgsz-jtxr-f3_a37,Details About Samsung 24 Class Hd (720p) Smart...,Samsung,UN24M4500AFXZA,"Samsung Tax Time Savings,TV, Video & Home Audi...",B073JLZ262,Walmart,138.00,189.99,True,True,27.36,2018-01-09
4,AVpgmxCxilAPnD_xuzi5,PELICAN - ProGear Case for Most Tablets - Black,PELICAN,1065-005-110,"Computers,Portfolio Cases tfqvddravsdxfrevbrca...",B008YGI7WK,Best Buy,59.99,59.99,False,False,0.00,2018-05-26
5,AVphxh7LilAPnD_x9K92,Details About Innovative Technology Itsbo358p ...,Innovative Technology,ITSBO-358P,"Portable Bluetooth Speakers,Stereos,TV, Video ...",B00UMVVU8S,Wholesale Connection,94.36,94.36,False,False,0.00,2018-03-19
6,AVphxh7LilAPnD_x9K92,Details About Innovative Technology Itsbo358p ...,Innovative Technology,ITSBO-358P,"Portable Bluetooth Speakers,Stereos,TV, Video ...",B00UMVVU8S,Wholesale Connection,94.36,94.36,False,False,0.00,2017-07-23
7,AVpfAXof1cnluZ0-bz3u,Samsung - 960 Pro 1TB Internal PCI Express 3.0...,Samsung,MZ-V6P1T0BW,"Computers,Hard Drives,Internal Drives,Solid St...",B01LYRCIPG,B&H Photo,629.99,629.99,False,False,0.00,2017-08-29
8,AVpivePGilAPnD_xHH0l,The Rebel BT On-Ear Wireless Bluetooth Headpho...,House of Marley,EMJH101BK,"Headphones,Bluetooth Headphones,All Headphones...",B01700RNIO,B&H Photo,36.00,36.00,False,False,0.00,2017-08-29
9,AVpffLmi1cnluZ0-lmL2,Rand McNally 0528011715 IntelliRoute 7 TND 730...,Rand McNally,528011715,"Electronics,GPS & Navigation,Handheld GPS Unit...",B00MBUZ2DW,Best Buy,282.99,282.99,False,False,0.00,2017-03-10


In [11]:
spark.sql("SELECT * FROM silver_electronics.amazon_ratings LIMIT 10").toPandas()

,asin,reviewer_id,rating,review_date
0,0446576476,A2WED9DICNJLKF,5,2005-11-05
1,0511189877,AHKMO8X1OE5T1,5,2016-09-26
2,0557348153,A3TUFHHA9A31J0,5,2013-04-11
3,073530498X,A1KFBWHT92P7NG,5,2016-07-11
4,0972683275,A11CLBU0YO7ARC,5,2014-10-23
5,0972683275,A2ZBZEI016H2V7,2,2014-07-31
6,0972683275,A29Y8MLKEZILVU,5,2013-10-02
7,0972683275,A14ZMA7222WU6F,5,2012-07-10
8,0972683275,A3R0H2RJY9DM99,5,2011-12-29
9,0972683275,A38DTJURWNUQTZ,5,2011-01-31


In [12]:
spark.sql("SELECT * FROM silver_electronics.amazon_products LIMIT 10").toPandas()

,asin,title,brand,main_cat,sub_category,category_path,price,release_date,sales_rank
0,0060219602,Hurray for Hattie Rabbit: Story and pictures (...,Visit Amazon's Dick Gackenbach Page,Books,eBook Readers & Accessories,Electronics > eBook Readers & Accessories > eB...,NaN,None,NaN
1,0101635370,abcGoodefg&reg; 4GB USB 2.0 Mp3 Music Player w...,Crazy Cart,All Electronics,Portable Audio & Video,Electronics > Portable Audio & Video > MP3 & M...,NaN,2012-12-28,177454.0
2,0132492776,Wireless Bluetooth Headphones Earbuds with Mic...,Enter The Arena,Home Audio & Theater,Headphones,Electronics > Headphones > Earbud Headphones,7.99,2017-10-23,950.0
3,0205291252,Practical Modern Basketball (3rd Edition),John R. Wooden,Books,Accessories & Supplies,Electronics > Accessories & Supplies,NaN,None,NaN
4,0285175270,"BOOX Max Carta Ereader,13.3"" Flexible Screen 1...",BOOX,All Electronics,eBook Readers & Accessories,Electronics > eBook Readers & Accessories > eB...,NaN,2016-06-30,264331.0
5,0312331363,Scared Money,James Hime,Books,eBook Readers & Accessories,Electronics > eBook Readers & Accessories > eB...,12.78,None,NaN
6,0441865704,Viriconium nights,Visit Amazon's M. John Harrison Page,Books,eBook Readers & Accessories,Electronics > eBook Readers & Accessories > eB...,NaN,None,NaN
7,0545811295,I Survived 10 Book Library,None,Home Audio & Theater,Accessories & Supplies,Electronics > Accessories & Supplies,NaN,2014-08-30,706381.0
8,0594017343,Barnes Noble Nook Simple Touch Reader BRNV300+...,Barnes and Noble,Cell Phones & Accessories,eBook Readers & Accessories,Electronics > eBook Readers & Accessories > eB...,NaN,None,1384.0
9,059403390X,Barnes & Noble BP-01-C01-N2-1 Industriell Cover,Barnes & Noble,Computers,Computers & Accessories,Electronics > Computers & Accessories > Tablet...,4.73,2011-01-09,87526.0


In [13]:
spark.sql("SELECT * FROM silver_electronics.prices_amazon_link LIMIT 10").toPandas()

,product_id,asin,match_type
0,AV0A-tRVGV-KLJ3aca3M,B01GATSH9U,asin
1,AV0A-tRVGV-KLJ3aca3M,B01H1BLHYS,asin
2,AV13C2saglJLPUi8O7pU,B001M4HF3S,asin
3,AV13ChPDvKc47QAVnh7U,B00XLZ3GD4,asin
4,AV13D7U_vKc47QAVni1h,B003S68Q0Y,asin
5,AV13EBmMvKc47QAVni6i,B00U90HMUC,asin
6,AV13EPfyGV-KLJ3akPCa,B01GL65NMQ,asin
7,AV13Evy9-jtxr-f38Bqf,B00CYSYZTS,asin
8,AV13GWKcGV-KLJ3akQXf,B00GQUGUKC,model_brand
9,AV13GijWglJLPUi8O-Df,B00630WKGI,asin


In [14]:
spark.sql("""
    SELECT MIN(date_seen) AS primeira_data, MAX(date_seen) AS ultima_data FROM silver_electronics.prices
""").toPandas()

,primeira_data,ultima_data
0,2014-08-30,2018-07-25


In [15]:
spark.sql("""
    SELECT merchant, COUNT(*) AS linhas
    FROM silver_electronics.prices
    GROUP BY merchant ORDER BY linhas DESC LIMIT 10
""").toPandas()

,merchant,linhas
0,Best Buy,15372
1,B&H Photo,9279
2,Walmart,2086
3,Beach Camera,570
4,None,249
5,buydig,213
6,Electronic Express,162
7,Focus Camera,161
8,AMI Ventures Inc,128
9,echo-and-optics,109


In [16]:
spark.sql("""
    SELECT is_sale, is_discount, COUNT(*) AS linhas
    FROM silver_electronics.prices
    GROUP BY is_sale, is_discount ORDER BY is_sale, is_discount
""").toPandas()

,is_sale,is_discount,linhas
0,False,False,27019
1,True,False,4738
2,True,True,1938


In [17]:
spark.sql("""
    SELECT rating, COUNT(*) AS avaliacoes
    FROM silver_electronics.amazon_ratings
    GROUP BY rating ORDER BY rating
""").toPandas()

,rating,avaliacoes
0,1,2370844
1,2,1117622
2,3,1498446
3,4,3231381
4,5,12349798


In [18]:
spark.sql("""
    SELECT l.match_type, p.name AS produto_datafiniti, a.title AS produto_amazon
    FROM silver_electronics.prices_amazon_link l
    JOIN (SELECT DISTINCT product_id, name FROM silver_electronics.prices) p ON l.product_id = p.product_id
    JOIN silver_electronics.amazon_products a ON l.asin = a.asin
    LIMIT 10
""").toPandas()

,match_type,produto_datafiniti,produto_amazon
0,asin,MartinLogan - Helos 12 6-1/2 In-Ceiling Speake...,MartinLogan Helos 12 In-Ceiling Speaker (White)
1,asin,Lumix G 25mm f/1.7 ASPH. Lens,Panasonic Leather Case for most of Lumix Slim ...
2,asin,Actiontec 300 Mbps Wireless-N ADSL Modem Route...,Actiontec 300 Mbps Wireless-N ADSL Modem Route...
3,asin,Nikon AF-S FX NIKKOR 200-500mm f/5.6E ED Vibra...,Nikon AF-S FX NIKKOR 200-500mm f/5.6E ED Vibra...
4,asin,RODE - NTG1 Condenser Shotgun Microphone,Rode NTG1 Condenser Shotgun Microphone
5,asin,"AW6500 All-Weather Outdoor Speaker (White, Sin...",Definitive Technology AW 6500 Outdoor Speaker ...
6,asin,Monster SuperStar BackFloat High-Definition Bl...,SuperStar 129279-00 BackFloat Bluetooth Speake...
7,asin,Alienware - R3 17.3 Laptop - Intel Core i7 - 1...,Alienware AW17R3-7092SLV 17.3-Inch FHD Laptop ...
8,asin,Sony STRDN1070 7.2-channel AV Receiver w/ Blue...,Sony 7.2 Channel Hi-Res Wi-Fi Network 4K AV Re...
9,asin,XD3 - 3 Active Desktop Monitor System (Pair),Cerwin Vega XD3 Powered Desktop Speakers


In [19]:
spark.sql("DESCRIBE HISTORY silver_electronics.prices").select("version", "timestamp", "operation").show(truncate=False)

+-------+-----------------------+------------+
|version|timestamp              |operation   |
+-------+-----------------------+------------+
|3      |2026-11-12 09:47:18.372|WRITE       |
|2      |2026-11-11 23:03:12.824|WRITE       |
|1      |2026-11-11 23:00:51.967|WRITE       |
|0      |2026-11-11 23:00:45.166|CREATE TABLE|
+-------+-----------------------+------------+



In [20]:
# Fecha a sessão spark
spark.stop()